# Scenario Dreamer Assets (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/scenario-dreamer-decision-layer/blob/main/notebooks/scenario_dreamer_assets_colab.ipynb)

This notebook prepares the **persistent Drive-backed baseline substrate**:
- sync this repo,
- clone and pin upstream Scenario Dreamer,
- bind canonical repo asset paths to Google Drive,
- optionally download the pretrained CtRL-Sim checkpoint,
- verify the checkpoint and Scenario Dreamer environment-pack locations.


In [ ]:
# Step 1: Sync this repo into the Colab runtime
import os
import subprocess
import sys

REPO_URL = 'https://github.com/achyutmorang/scenario-dreamer-decision-layer.git'
REPO_DIR = '/content/scenario-dreamer-decision-layer'

if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '-b', 'main', REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', REPO_DIR], check=True)
for p in (REPO_DIR, os.path.join(REPO_DIR, 'src')):
    if p not in sys.path:
        sys.path.insert(0, p)

print('repo_rev:', subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD'], text=True).strip())


In [ ]:
# Suggested defaults: Drive-backed persistent asset layout.
%env SCENARIO_DREAMER_DRIVE_ROOT=/content/drive/MyDrive/scenario_dreamer_research
%env SCENARIO_DREAMER_DOWNLOAD_CHECKPOINT=0


In [ ]:
# Step 2: Mount Drive and inspect the default persistent layout
import json
import os
from pathlib import Path

from google.colab import drive
from scenario_dreamer_decision_layer.colab import default_drive_layout

drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT = os.environ['SCENARIO_DREAMER_DRIVE_ROOT']
layout = {k: str(v) for k, v in default_drive_layout(DRIVE_ROOT).items()}
print(json.dumps(layout, indent=2, sort_keys=True))


In [ ]:
# Step 3: Clone/pin upstream Scenario Dreamer and bind canonical repo asset paths to Drive
import json
import os
import subprocess
import sys

from scenario_dreamer_decision_layer.colab import bind_drive_layout, inspect_bound_layout

subprocess.run([sys.executable, 'scripts/bootstrap_baseline.py', '--clone-upstream', '--write-lock'], check=True)
binding = bind_drive_layout(os.environ['SCENARIO_DREAMER_DRIVE_ROOT'])
print('binding:')
print(json.dumps(binding, indent=2, sort_keys=True))
print('inspection:')
print(json.dumps(inspect_bound_layout(), indent=2, sort_keys=True))


In [ ]:
# Step 4: Optional checkpoint download into Drive-backed canonical scratch_root
import os
import subprocess
import sys

DOWNLOAD_CHECKPOINT = os.environ.get('SCENARIO_DREAMER_DOWNLOAD_CHECKPOINT', '0').strip().lower() in {'1', 'true', 'yes', 'y', 'on'}
print('DOWNLOAD_CHECKPOINT:', DOWNLOAD_CHECKPOINT)
if DOWNLOAD_CHECKPOINT:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'gdown'], check=True)
    subprocess.run([sys.executable, 'scripts/download_assets.py', '--mode', 'checkpoint', '--download'], check=True)
else:
    print('Skipping checkpoint download. Set SCENARIO_DREAMER_DOWNLOAD_CHECKPOINT=1 to enable it.')


In [ ]:
# Step 5: Verify current asset state
import subprocess
import sys

subprocess.run([sys.executable, 'scripts/download_assets.py', '--mode', 'all'], check=True)
print('
If the environment pack is still missing, place the released Scenario Dreamer Waymo pickles and jsons into the Drive paths printed above.')
